# Master Thesis Experiments

Notebook for the experiment log, setup steps, and result notes for the master thesis.


## E0 - Valid max trial step count

Goal:
- estimate a reasonable maximum number of trial steps for successful rollouts
- use a deterministic evaluation setup so the step distribution is reproducible
- sample a fixed subset of anatomies first, then reuse that subset for the experiment

Configuration:
- `policy_mode = deterministic`
- `stochastic_environment_mode = random_start`
- `max_episode_steps = 1000`
- `n_anatomies = 12`
- `targets = 3 random targets per anatomy`
- `trials_per_cell = 100`


### E0.0 Notebook imports and project paths

This section defines the project root once, adds the helper module to `sys.path`, and imports the utilities used by E0.1-E0.4. It is intentionally placed at the top so the later cells stay short and do not depend on a fragile relative working directory.


In [11]:
from pathlib import Path
from typing import Optional
import json
import sys

from IPython.display import Markdown, display


def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        helper = candidate / 'experiments' / 'master-thesis' / 'notebook_helpers' / 'e0.py'
        if helper.exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the project root from the current notebook working directory.')


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / 'experiments' / 'master-thesis' / 'notebook_helpers'
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

from e0 import (
    DEFAULT_ANALYSIS_ROOT,
    DEFAULT_RESULTS_ROOT,
    analyze_e0_results,
    build_chunked_runner_script,
    load_e0_results,
    split_sample_json_into_chunks,
)
from steve_recommender.eval_v2.experimental_prep_scripts.sample_anatomies import sample_anatomies_from_registry
from steve_recommender.eval_v2.service import DefaultEvaluationService

print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'HELPER_DIR={HELPER_DIR}')


PROJECT_ROOT=/home/hannes-deittert/dev/Uni/master-project
HELPER_DIR=/home/hannes-deittert/dev/Uni/master-project/experiments/master-thesis/notebook_helpers


from IPython.display import Markdown, display

POOL = PROJECT_ROOT / 'data' / 'anatomy_registry'
OUT = PROJECT_ROOT / 'results' / 'experimental_prep' / 'sample_12.json'
POOL_INDEX = POOL / 'index.json'

payload = sample_anatomies_from_registry(
    pool_path=POOL,
    n=12,
    seed=123,
    strata='none',
    sampling_method='random',
    branches=('bct', 'lcca', 'lsa'),
    workers=4,
)

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '
', encoding='utf-8')

service = DefaultEvaluationService()
rows = []
for item in payload['selected_anatomies']:
    anatomy = service.get_anatomy(record_id=item['record_id'], registry_path=POOL_INDEX)
    branches = service.list_branches(anatomy)
    rows.append(
        {
            'record_id': item['record_id'],
            'arch_type': item['arch_type'],
            'seed': item['seed'],
            'target_branches': ', '.join(branch.name for branch in branches),
            'tortuosity': item['tortuosity'],
            'stratum': item['stratum'],
        }
    )

table_lines = [
    '| record_id | arch_type | seed | target_branches | tortuosity | stratum |',
    '|---|---|---:|---|---:|---|',
]
for row in rows:
    table_lines.append(
        '| {record_id} | {arch_type} | {seed} | {target_branches} | {tortuosity:.6f} | {stratum} |'.format(**row)
    )

display(Markdown('
'.join(table_lines)))
print(json.dumps(rows, indent=2, sort_keys=True))


In [12]:
POOL = PROJECT_ROOT / 'data' / 'anatomy_registry'
OUT = PROJECT_ROOT / 'results' / 'experimental_prep' / 'sample_12.json'
POOL_INDEX = POOL / 'index.json'

payload = sample_anatomies_from_registry(
    pool_path=POOL,
    n=12,
    seed=123,
    strata='none',
    sampling_method='random',
    branches=('bct', 'lcca', 'lsa'),
    workers=4,
)

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

service = DefaultEvaluationService()
rows = []
for item in payload['selected_anatomies']:
    anatomy = service.get_anatomy(record_id=item['record_id'], registry_path=POOL_INDEX)
    branches = service.list_branches(anatomy)
    rows.append(
        {
            'record_id': item['record_id'],
            'arch_type': item['arch_type'],
            'seed': item['seed'],
            'target_branches': ', '.join(branch.name for branch in branches),
            'tortuosity': item['tortuosity'],
            'stratum': item['stratum'],
        }
    )

table_lines = [
    '| record_id | arch_type | seed | target_branches | tortuosity | stratum |',
    '|---|---|---:|---|---:|---|',
]
for row in rows:
    table_lines.append(
        '| {record_id} | {arch_type} | {seed} | {target_branches} | {tortuosity:.6f} | {stratum} |'.format(**row)
    )

display(Markdown('\n'.join(table_lines)))
print(json.dumps(rows, indent=2, sort_keys=True))


| record_id | arch_type | seed | target_branches | tortuosity | stratum |
|---|---|---:|---|---:|---|
| Tree_625 | I | 1248887895 | aorta, bct, lcca, lsa, rcca, rsa | 0.018721 | all |
| Tree_857 | I | 659745269 | aorta, bct, lcca, lsa, rcca, rsa | 0.027128 | all |
| Tree_1428 | I | 2105833333 | aorta, bct, lcca, lsa, rcca, rsa | 0.015694 | all |
| Tree_1764 | I | 1577845490 | aorta, bct, lcca, lsa, rcca, rsa | 0.016744 | all |
| Tree_4367 | I | 57901021 | aorta, bct, lcca, lsa, rcca, rsa | 0.027405 | all |
| Tree_4385 | I | 1671607265 | aorta, bct, lcca, lsa, rcca, rsa | 0.022011 | all |
| Tree_5442 | I | 503907874 | aorta, bct, lcca, lsa, rcca, rsa | 0.024660 | all |
| Tree_5583 | I | 797089154 | aorta, bct, lcca, lsa, rcca, rsa | 0.024165 | all |
| Tree_6211 | I | 1096800833 | aorta, bct, lcca, lsa, rcca, rsa | 0.028689 | all |
| Tree_6672 | I | 1288778203 | aorta, bct, lcca, lsa, rcca, rsa | 0.027289 | all |
| Tree_8785 | I | 1291750865 | aorta, bct, lcca, lsa, rcca, rsa | 0.024043 | all |
| Tree_9213 | I | 908489492 | aorta, bct, lcca, lsa, rcca, rsa | 0.029544 | all |

[
  {
    "arch_type": "I",
    "record_id": "Tree_625",
    "seed": 1248887895,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.018721201833
  },
  {
    "arch_type": "I",
    "record_id": "Tree_857",
    "seed": 659745269,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.027127724525
  },
  {
    "arch_type": "I",
    "record_id": "Tree_1428",
    "seed": 2105833333,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.015694332837
  },
  {
    "arch_type": "I",
    "record_id": "Tree_1764",
    "seed": 1577845490,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.016743728769
  },
  {
    "arch_type": "I",
    "record_id": "Tree_4367",
    "seed": 57901021,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.027405191123
  },
  {
    

SAMPLE_JSON = PROJECT_ROOT / 'results' / 'experimental_prep' / 'sample_12.json'
rows = split_sample_json_into_chunks(SAMPLE_JSON, chunk_size=3, output_dir=SAMPLE_JSON.parent)

lines = ['| chunk_id | n_anatomies | record_ids | chunk_path |', '|---|---:|---|---|']
for row in rows:
    lines.append(
        f"| {row['chunk_id']} | {row['n_anatomies']} | {row['record_ids']} | {row['chunk_path']} |"
    )

table_md = '\n'.join(lines)
display(Markdown(table_md))
print(table_md)


In [13]:
from IPython.display import Markdown, display

SAMPLE_JSON = PROJECT_ROOT / 'results' / 'experimental_prep' / 'sample_12.json'
rows = split_sample_json_into_chunks(SAMPLE_JSON, chunk_size=3, output_dir=SAMPLE_JSON.parent)

lines = ['| chunk_id | n_anatomies | record_ids | chunk_path |', '|---|---:|---|---|']
for row in rows:
    lines.append(
        f"| {row['chunk_id']} | {row['n_anatomies']} | {row['record_ids']} | {row['chunk_path']} |"
    )

table_md = '\n'.join(lines)
display(Markdown(table_md))
print(table_md)


ModuleNotFoundError: No module named 'pandas'

script_path = PROJECT_ROOT / 'scripts' / 'e0_chunks.tinygpu.sbatch'
script_text = build_chunked_runner_script(
    PROJECT_ROOT,
    array_size=4,
    partition='work',
    gpus=4,
    cpus_per_task=32,
    walltime='24:00:00',
    worker_count=29,
    runs_per_anatomy=3,
    trial_count=100,
    max_episode_steps=1000,
    base_seed_start=123,
    target_seed_start=9000,
    base_seed_step=200,
)
script_path.parent.mkdir(parents=True, exist_ok=True)
script_path.write_text(script_text, encoding='utf-8')
print(script_path.read_text(encoding='utf-8'))


In [ ]:
script_path = PROJECT_ROOT / 'scripts' / 'e0_chunks.tinygpu.sbatch'
script_text = build_chunked_runner_script(
    PROJECT_ROOT,
    array_size=4,
    partition='work',
    gpus=4,
    cpus_per_task=32,
    walltime='24:00:00',
    worker_count=29,
    runs_per_anatomy=3,
    trial_count=100,
    max_episode_steps=1000,
    base_seed_start=123,
    target_seed_start=9000,
    base_seed_step=200,
)
script_path.parent.mkdir(parents=True, exist_ok=True)
script_path.write_text(script_text, encoding='utf-8')
print(script_path.read_text(encoding='utf-8'))


RESULT_ROOT = Path(os.environ.get('E0_RESULTS_ROOT', str(DEFAULT_RESULTS_ROOT))).resolve()
ANALYSIS_ROOT = DEFAULT_ANALYSIS_ROOT
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'RESULT_ROOT={RESULT_ROOT}')
print(f'ANALYSIS_ROOT={ANALYSIS_ROOT}')

manifest_df, summary_df, trial_df = load_e0_results(RESULT_ROOT)
analysis = analyze_e0_results(trial_df, analysis_root=ANALYSIS_ROOT)

print(f'manifests={len(manifest_df)}')
print(f'candidate_rows={len(summary_df)}')
print(f'trial_rows={len(trial_df)}')

display(Markdown('#### Run coverage'))
display(manifest_df[['job_name', 'chunk_dir', 'n_anatomies', 'n_trials_total']].sort_values(['chunk_dir', 'job_name']))

display(Markdown('#### Candidate summary'))
display(analysis['candidate_agg'])

if not analysis['percentiles'].empty:
    display(Markdown('#### Step-budget percentiles for successful trials'))
    display(analysis['percentiles'].reset_index().rename(columns={'index': 'quantile'}).assign(quantile=lambda df: df['quantile'].map(lambda x: f'{int(x * 100)}%')))
    summary_lines = [
        f"- current max_episode_steps: `{analysis['current_max']}`",
        f"- suggested 95th-percentile cutoff: `{analysis['recommended_cutoff']}` steps",
        f"- successful trials analyzed: `{len(analysis['successful_steps'])}`",
    ]
    display(Markdown('\n'.join(summary_lines)))
else:
    display(Markdown('**No successful trials were found yet.**'))

for image_name in [
    'e0_candidate_summary.png',
    'e0_step_budget_curve.png',
    'e0_episode_length_ecdf.png',
]:
    image_path = ANALYSIS_ROOT / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))
        display(Markdown(f'*Saved:* `{image_path}`'))


In [ ]:
from IPython.display import Image, Markdown, display
import os

RESULT_ROOT = Path(os.environ.get('E0_RESULTS_ROOT', str(DEFAULT_RESULTS_ROOT))).resolve()
ANALYSIS_ROOT = DEFAULT_ANALYSIS_ROOT
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'RESULT_ROOT={RESULT_ROOT}')
print(f'ANALYSIS_ROOT={ANALYSIS_ROOT}')

try:
    manifest_rows, summary_rows, trial_rows = load_e0_results(RESULT_ROOT)
except FileNotFoundError:
    display(Markdown(
        '### No synced results found yet\n'
        f'- expected root: `{RESULT_ROOT}`\n'
        '- sync the remote `results/master_thesis/e0_chunks/` tree here, preserving the per-chunk subfolders and `manifest.json` files\n'
        '- or set `E0_RESULTS_ROOT` to the synced directory before rerunning this cell'
    ))
else:
    analysis = analyze_e0_results(trial_rows, analysis_root=ANALYSIS_ROOT)

    print(f'manifests={len(manifest_rows)}')
    print(f'candidate_rows={len(summary_rows)}')
    print(f'trial_rows={len(trial_rows)}')

    def md_table(rows, columns, formatters=None):
        formatters = formatters or {}
        lines = ['| ' + ' | '.join(columns) + ' |', '|'+ '|'.join(['---'] * len(columns)) + '|']
        for row in rows:
            values = []
            for column in columns:
                value = row.get(column, '')
                formatter = formatters.get(column)
                if formatter is not None:
                    value = formatter(value)
                elif value is None:
                    value = ''
                values.append(str(value))
            lines.append('| ' + ' | '.join(values) + ' |')
        return '\n'.join(lines)

    display(Markdown('#### Run coverage'))
    display(Markdown(md_table(
        manifest_rows,
        ['job_name', 'chunk_dir', 'n_anatomies', 'n_trials_total'],
    )))

    display(Markdown('#### Candidate summary'))
    display(Markdown(md_table(
        analysis['candidate_rows'],
        ['execution_wire', 'n_trials', 'success_rate', 'score_mean', 'score_std', 'steps_total_mean', 'steps_total_p95', 'steps_to_success_mean', 'steps_to_success_p95'],
        formatters={
            'success_rate': lambda v: '' if v is None else f'{float(v):.3f}',
            'score_mean': lambda v: '' if v is None else f'{float(v):.3f}',
            'score_std': lambda v: '' if v is None else f'{float(v):.3f}',
            'steps_total_mean': lambda v: '' if v is None else f'{float(v):.1f}',
            'steps_total_p95': lambda v: '' if v is None else f'{float(v):.1f}',
            'steps_to_success_mean': lambda v: '' if v is None else f'{float(v):.1f}',
            'steps_to_success_p95': lambda v: '' if v is None else f'{float(v):.1f}',
        },
    )))

    if analysis['percentiles_rows']:
        display(Markdown('#### Step-budget percentiles for successful trials'))
        display(Markdown(md_table(
            analysis['percentiles_rows'],
            ['quantile', 'steps_to_success'],
            formatters={'steps_to_success': lambda v: '' if v is None else f'{float(v):.1f}'},
        )))
        summary_lines = [
            f"- current max_episode_steps: `{analysis['current_max']}`",
            f"- suggested 95th-percentile cutoff: `{analysis['recommended_cutoff']}` steps",
            f"- successful trials analyzed: `{len(analysis['successful_steps'])}`",
        ]
        display(Markdown('\n'.join(summary_lines)))
    else:
        display(Markdown('**No successful trials were found yet.**'))

    for image_name in [
        'e0_candidate_summary.png',
        'e0_step_budget_curve.png',
        'e0_episode_length_ecdf.png',
    ]:
        image_path = ANALYSIS_ROOT / image_name
        if image_path.exists():
            display(Image(filename=str(image_path)))
            display(Markdown(f'*Saved:* `{image_path}`'))


### E0.4 Answer field

- Recommended `max_episode_steps`: `________`
- Evidence summary: `________`
- Follow-up action: `________`


## E1 - Evaluation score design and cluster orchestration

E1 is the calibration experiment that decides which evaluation setting we use for the main thesis runs. The design is intentionally simple at the Slurm layer and strict at the seed layer:

- 4 configs: deterministic/stochastic policy, fixed/random start
- 12 anatomies
- 3 targets per anatomy, shared across all 4 configs
- 15 wires are evaluated locally inside each Slurm job
- 1 Slurm job = 1 `(config, anatomy, target)` triple
- the manifest carries the step budget as `max_episode_steps`

For smoke tests, reduce the anatomy set to 1 and override `trial_count` to 1. For the full run, keep the 12-anatomy sample and let the manifest carry the real trial count and step budget.


In [14]:
from pathlib import Path

E1_SAMPLE_SCRIPT = PROJECT_ROOT / "scripts" / "sample_e1_anatomies.py"
E1_SAMPLE_JSON = PROJECT_ROOT / "results" / "experimental_prep" / "sample_12_e1.json"
E1_RESULTS_ROOT = PROJECT_ROOT / "results" / "master_thesis" / "e1"

print(f"E1_SAMPLE_SCRIPT={E1_SAMPLE_SCRIPT}")
print(f"E1_SAMPLE_JSON={E1_SAMPLE_JSON}")
print(f"E1_RESULTS_ROOT={E1_RESULTS_ROOT}")


E1_SAMPLE_SCRIPT=/home/hannes-deittert/dev/Uni/master-project/scripts/sample_e1_anatomies.py
E1_SAMPLE_JSON=/home/hannes-deittert/dev/Uni/master-project/results/experimental_prep/sample_12_e1.json
E1_RESULTS_ROOT=/home/hannes-deittert/dev/Uni/master-project/results/master_thesis/e1


### 1. Sample 12 anatomies for E1

This writes a fresh 12-anatomy input set for E1 without touching the E0 sample. The script is reproducible because it uses a fixed seed.

```bash
python3 scripts/sample_e1_anatomies.py \
  --output results/experimental_prep/sample_12_e1.json \
  --n 12 \
  --seed 123 \
  --strata none \
  --sampling-method random \
  --branches bct,lcca,lsa \
  --workers 4
```

If you want a smaller smoke file, sample only one anatomy and keep the same target generation logic:

```bash
python3 scripts/sample_e1_anatomies.py --output results/experimental_prep/sample_1.json --n 1 --seed 123
```


In [15]:
POOL = PROJECT_ROOT / 'data' / 'anatomy_registry'
OUT = PROJECT_ROOT / 'results' / 'experimental_prep' / 'sample_12_e1.json'
POOL_INDEX = POOL / 'index.json'

payload = sample_anatomies_from_registry(
    pool_path=POOL,
    n=12,
    seed=456,  # different from E0
    strata='none',
    sampling_method='random',
    branches=('bct', 'lcca', 'lsa'),
    workers=4,
)

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

service = DefaultEvaluationService()
rows = []
for item in payload['selected_anatomies']:
    anatomy = service.get_anatomy(record_id=item['record_id'], registry_path=POOL_INDEX)
    branches = service.list_branches(anatomy)
    rows.append(
        {
            'record_id': item['record_id'],
            'arch_type': item['arch_type'],
            'seed': item['seed'],
            'target_branches': ', '.join(branch.name for branch in branches),
            'tortuosity': item['tortuosity'],
            'stratum': item['stratum'],
        }
    )

table_lines = [
    '| record_id | arch_type | seed | target_branches | tortuosity | stratum |',
    '|---|---|---:|---|---:|---|',
]
for row in rows:
    table_lines.append(
        '| {record_id} | {arch_type} | {seed} | {target_branches} | {tortuosity:.6f} | {stratum} |'.format(**row)
    )

display(Markdown('\n'.join(table_lines)))
print(json.dumps(rows, indent=2, sort_keys=True))


| record_id | arch_type | seed | target_branches | tortuosity | stratum |
|---|---|---:|---|---:|---|
| Tree_660 | I | 1986021659 | aorta, bct, lcca, lsa, rcca, rsa | 0.017733 | all |
| Tree_1382 | I | 1330280170 | aorta, bct, lcca, lsa, rcca, rsa | 0.022330 | all |
| Tree_1882 | I | 667750211 | aorta, bct, lcca, lsa, rcca, rsa | 0.021278 | all |
| Tree_1996 | I | 1895465938 | aorta, bct, lcca, lsa, rcca, rsa | 0.016279 | all |
| Tree_5348 | I | 349931947 | aorta, bct, lcca, lsa, rcca, rsa | 0.014258 | all |
| Tree_6399 | I | 931406019 | aorta, bct, lcca, lsa, rcca, rsa | 0.019328 | all |
| Tree_7130 | I | 36334367 | aorta, bct, lcca, lsa, rcca, rsa | 0.020217 | all |
| Tree_7363 | I | 1192301222 | aorta, bct, lcca, lsa, rcca, rsa | 0.015256 | all |
| Tree_7638 | I | 2023377108 | aorta, bct, lcca, lsa, rcca, rsa | 0.018607 | all |
| Tree_7841 | I | 353179492 | aorta, bct, lcca, lsa, rcca, rsa | 0.015801 | all |
| Tree_8010 | I | 995137380 | aorta, bct, lcca, lsa, rcca, rsa | 0.021043 | all |
| Tree_8547 | I | 711060338 | aorta, bct, lcca, lsa, rcca, rsa | 0.029898 | all |

[
  {
    "arch_type": "I",
    "record_id": "Tree_660",
    "seed": 1986021659,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.017732588784
  },
  {
    "arch_type": "I",
    "record_id": "Tree_1382",
    "seed": 1330280170,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.02232958738
  },
  {
    "arch_type": "I",
    "record_id": "Tree_1882",
    "seed": 667750211,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.021278029057
  },
  {
    "arch_type": "I",
    "record_id": "Tree_1996",
    "seed": 1895465938,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.016278766868
  },
  {
    "arch_type": "I",
    "record_id": "Tree_5348",
    "seed": 349931947,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.01425790713
  },
  {
    

### 2. Sync the E1 files to TinyGPU

Sync the helpers, the generator, the runner, and the validation scripts to the cluster before submitting anything. Keep `--relative` so the paths land in the same layout on the cluster.

```bash
rsync -av --relative \
  experiments/master-thesis/notebook_helpers/e1.py \
  experiments/master-thesis/generate_e1_jobs.py \
  experiments/master-thesis/run_e1_cell.py \
  scripts/sample_e1_anatomies.py \
  scripts/e1_generate_validate.py \
  scripts/e1_partition_probe.py \
  scripts/e1_analyze_manifest.py \
  tests/e1/ \
  iwhr106h@tinyx.nhr.fau.de:/home/woody/iwhr/iwhr106h/master-project/
```

If you changed only the notebook, you do not need to sync the generated `results/` tree. For a real submission, sync the updated `results/experimental_prep/sample_12_e1.json` if you generated it locally.


### 3. Generate and validate the manifest

Use the validation wrapper to probe the live cluster, derive partition weights, generate the bucketed Slurm scripts, and verify that the same targets are reused across all 4 configs.

```bash
python3 scripts/e1_generate_validate.py \
  --sample-json results/experimental_prep/sample_12_e1.json \
  --output-root results/master_thesis/e1 \
  --probe-cluster \
  --max-episode-steps 1000
```

Useful variants:

```bash
# Force a single partition instead of probing the cluster
python3 scripts/e1_generate_validate.py --sample-json results/experimental_prep/sample_12_e1.json --output-root results/master_thesis/e1 --partitions work --max-episode-steps 1000

# True smoke run: one anatomy, one target, one trial per config
python3 scripts/e1_generate_validate.py \
  --sample-json results/experimental_prep/sample_1.json \
  --output-root results/master_thesis/e1_smoke_tiny \
  --target-count-per-anatomy 1 \
  --trial-count 1 \
  --partitions work \
  --max-episode-steps 50
```

The partition probe uses the current TinyGPU state. In the current heuristic, the weight for a partition is proportional to `idle_nodes + 0.5 * mix_nodes`. If no usable partition is found, the generator falls back to `work`.


### 4. Submit the cluster jobs

The validation wrapper writes one sbatch file per selected partition. Submit the generated scripts directly. The exact set depends on the probe result; `work` is always valid, while `a100`, `rtx3080`, or `v100` only appear if the live cluster allocation gives them weight.

```bash
sbatch.tinygpu scripts/e1_work.sbatch
sbatch.tinygpu scripts/e1_a100.sbatch
sbatch.tinygpu scripts/e1_rtx3080.sbatch
```

For direct debugging of a single generated row, use the runner and pass a step budget explicitly:

```bash
python3 experiments/master-thesis/run_e1_cell.py \
  --manifest results/master_thesis/e1/metadata/job_manifest_work.json \
  --array-index 0 \
  --step-budget 1000
```

The names to remember are:

- `--max-episode-steps` on `scripts/e1_generate_validate.py` and `generate_e1_jobs.py`
- `--step-budget` on `experiments/master-thesis/run_e1_cell.py`


### 5. E1 answer field

- Recommended `max_episode_steps`: `________`
- Partition choice for the full run: `________`
- Evidence summary: `________`


E1 "how should we constitute an evaluation score?"
Config 1: Deterministic, N=1. One trial per evaluation. Policy is deterministic, environment is whatever it is at start. Score is just that one trial's score. No aggregation.
Config 2: Deterministic, N=k, varied env_seed. k trials, all with deterministic policy but different env_seed per trial. Aggregate (IQM) over the k trials. The variance in the aggregate comes from environment perturbations alone.
Config 3: Stochastic, N=k, fixed env_seed. k trials, all with the same physical starting state but different policy_seed. Aggregate over the k trials. Variance comes from action sampling alone.
Config 4: Stochastic, N=k, varied env_seed. k trials, both policy and environment seeds vary. Variance from both sources combined.




G1. Establish whether stEVE-simulated RL expert agents constitute a stable measurement system whose outputs (a) support a methodology for clinically-grounded guidewire ranking, and (b) function as a benchmark environment for evaluating autonomous endovascular navigation methods on clinical outcomes rather than success rate alone, by characterising the system's stability under stochastic perturbation, its discriminability across wires, and its sensitivity to the reward function used during agent training.


H1 — Stability (within-anatomy ranking reliability).
The wire ranking produced by the system on a given anatomy is stable across stochastic perturbation. Operationally: Kendall's τ between rankings produced from independent seed splits exceeds a pre-registered threshold (suggested: τ ≥ 0.7) for the majority of anatomies tested.
H1 falsified ⟹ Contribution 1 substantially weakened (you can't recommend a wire if the ranking flips between runs); Contribution 2 partially weakened (a noisy benchmark is still useful but less so).


H2 — Discriminability (the wires are actually distinguishable).
The 15 wires resolve into multiple statistically distinguishable tiers under the eval scoring. Operationally: pairwise probability-of-improvement matrix shows ≥ k tiers (pre-register k = 3) with within-tier P(A > B) near 0.5 and between-tier P > 0.75.
H2 falsified ⟹ either the wire matrix is too narrow to differentiate (Contribution 1 has limited scope but isn't wrong), or the eval metric is too coarse (Contribution 2 needs work). Diagnostic in either direction.


H3 — Reward sensitivity (the expert-agent assumption holds, or doesn't).
Rankings produced by task-reward agents (Set A) and force-aware agents (Set B) on the same anatomies and targets agree above a pre-registered threshold (suggested: τ ≥ 0.6 between ranking pairs).
H3 confirmed ⟹ rankings are robust to reward choice; the expert-agent assumption is approximately correct. Contribution 1 strengthened. Contribution 2 still useful but less differentiating.
H3 falsified ⟹ rankings depend on training reward in a structured way. This is a finding, not a failure: Contribution 1 must be reframed as "ranking under specified reward" rather than "the ranking of the wire," and Contribution 2 is strengthened because the benchmark can detect reward-design differences.
Either outcome is publishable. This is what we want from a hypothesis.


H4 — Cross-anatomy structure (does anatomy matter, and how).
Cross-anatomy ranking similarity (the τ-matrix between anatomies' rankings) correlates with anatomical features in a mechanically interpretable way. Specifically: anatomies similar in tortuosity / branch geometry produce similar rankings; anatomies dissimilar in those features produce dissimilar rankings.


H4 confirmed ⟹ the system captures real mechanical structure; anatomy-conditional recommendation is in principle feasible (future work).
H4 partially confirmed (some structure but noisy) ⟹ the methodology has resolution limits at fine anatomical distinctions; useful contribution but bounded.
H4 falsified (rankings vary randomly across anatomies) ⟹ either anatomy descriptors are wrong, or the system isn't capturing mechanics, or both. Diagnostic.
H4 is where your old H1/H2/H3 mechanistic content lives — but now as explanatory analysis of the cross-anatomy structure, not as primary claims.
